In [22]:
!pip install pandas matplotlib

In [23]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

report_path = Path("banco.report.json")   # mismo directorio
report = json.loads(report_path.read_text(encoding="utf-8"))

print("Modelo:", report["run"]["model"])
print("Seeds:", report["run"]["seeds_found"])
print("Preguntas exportadas:", report["run"]["questions_exported"])

thr = report.get("thresholds", {})
qdf = pd.DataFrame(report["questions"])
qdf.head()

Modelo: gpt-4o-mini
Seeds: 30
Preguntas exportadas: 60


,question_id,seed_id,type,difficulty,answer_type,warnings,jaccard_distractor_correct,jaccard_distractor_stem,choice_len_ratio,rationale_lengths
0,dns-what-is-q1,dns-what-is,mcq,media,funcion,[WARNING: distractor poco relacionado con el e...,"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",1.500000,"[68, 54, 51]"
1,dns-what-is-q2,dns-what-is,mcq,media,definicion,[WARNING: distractor poco relacionado con el e...,"[0.5, 0.0, 0.0]","[0.0, 0.0, 0.0]",1.161290,"[50, 60, 49]"
2,dns-hosts-file-q1,dns-hosts-file,mcq,media,funcion,[WARNING: distractor poco relacionado con el e...,"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.1]",2.380952,"[84, 69, 108]"
3,dns-hosts-file-q2,dns-hosts-file,mcq,media,definicion,[WARNING: distractor poco relacionado con el e...,"[0.5, 0.0, 0.0]","[0.0, 0.0, 0.0]",1.480000,"[89, 83, 88]"
4,dns-hierarchy-q1,dns-hierarchy,mcq,media,definicion,[WARNING: distractor poco relacionado con el e...,"[0.14285714285714285, 0.14285714285714285, 0.1...","[0.0, 0.0, 0.0]",2.300000,"[47, 67, 47]"


In [24]:
figs_dir = Path("figs")
figs_dir.mkdir(exist_ok=True)
figs_dir

WindowsPath('figs')

In [25]:
def flatten_list_column(df, col):
    vals = []
    if col not in df.columns:
        return vals
    for x in df[col].dropna().tolist():
        if isinstance(x, list):
            vals.extend([v for v in x if v is not None])
    return vals

In [26]:
vals = flatten_list_column(qdf, "jaccard_distractor_correct")

plt.figure()
plt.hist(vals, bins=20)
plt.axvline(thr["jaccard_distractor_correct_max"])
plt.title("Similitud Jaccard: distractor vs correcta")
plt.xlabel("Jaccard")
plt.ylabel("Frecuencia")
out = figs_dir / "fig_4_4_jaccard_distractor_correct.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Guardada:", out)

Guardada: figs\fig_4_4_jaccard_distractor_correct.png


In [27]:
vals = flatten_list_column(qdf, "jaccard_distractor_stem")

plt.figure()
plt.hist(vals, bins=20)
plt.axvline(thr["jaccard_distractor_stem_min"])
plt.title("Similitud Jaccard: distractor vs enunciado")
plt.xlabel("Jaccard")
plt.ylabel("Frecuencia")
out = figs_dir / "fig_4_5_jaccard_distractor_stem.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Guardada:", out)

Guardada: figs\fig_4_5_jaccard_distractor_stem.png


In [28]:
def count_warnings(x):
    return len(x) if isinstance(x, list) else 0

qdf["n_warnings"] = qdf["warnings"].apply(count_warnings)

plt.figure()
plt.hist(qdf["n_warnings"], bins=range(int(qdf["n_warnings"].max()) + 2))
plt.title("Número de warnings por pregunta")
plt.xlabel("Warnings por pregunta")
plt.ylabel("Frecuencia")
out = figs_dir / "fig_4_6_warnings_per_question.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Guardada:", out)

Guardada: figs\fig_4_6_warnings_per_question.png


In [29]:
if "choice_len_ratio" in qdf.columns:
    vals = qdf["choice_len_ratio"].dropna().tolist()
else:
    vals = []

plt.figure()
plt.hist(vals, bins=20)
plt.axvline(thr["choice_length_ratio_max"])
plt.title("Desbalance de longitudes entre opciones (max/min)")
plt.xlabel("Ratio max/min")
plt.ylabel("Frecuencia")
out = figs_dir / "fig_4_7_choice_length_ratio.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Guardada:", out)

Guardada: figs\fig_4_7_choice_length_ratio.png


In [30]:
vals = flatten_list_column(qdf, "rationale_lengths")

plt.figure()
plt.hist(vals, bins=20)
plt.axvline(thr["rationale_min_chars"])
plt.title("Longitud de distractor_rationales")
plt.xlabel("Caracteres")
plt.ylabel("Frecuencia")
out = figs_dir / "fig_4_8_rationale_lengths.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Guardada:", out)

Guardada: figs\fig_4_8_rationale_lengths.png


In [31]:
print("\nResumen")
print("- Preguntas:", report["run"]["questions_exported"])
print("- Warnings totales:", int(qdf["n_warnings"].sum()))
print("- Media warnings/pregunta:", float(qdf["n_warnings"].mean()) if len(qdf) else 0.0)
qdf[["question_id","seed_id","n_warnings"]].sort_values("n_warnings", ascending=False).head(10)


Resumen
- Preguntas: 60
- Warnings totales: 131
- Media warnings/pregunta: 2.183333333333333


,question_id,seed_id,n_warnings
59,dns-essential-commands-q2,dns-essential-commands,4
58,dns-essential-commands-q1,dns-essential-commands,4
40,dns-tools-dig-nslookup-q1,dns-tools-dig-nslookup,4
0,dns-what-is-q1,dns-what-is,3
6,dns-components-q1,dns-components,3
18,dns-forward-reverse-q1,dns-forward-reverse,3
13,dns-zones-q2,dns-zones,3
26,bind-install-q1,bind-install,3
17,dns-soa-record-q2,dns-soa-record,3
1,dns-what-is-q2,dns-what-is,3
